##### CORE REDIS
- Data Types : Redis can store strings, Hashes, lists, sets.
- TTl & Expiration : Time to live & expiration, basically how long will data live in memory.
- Persitance : It means saving data in disk so it can survive a server restart. Methods - RDB & AOF
- Pipeline : It execute group of commands  togathe, reducing network round trips.
- Transaction : - Ensures block of commands execute without other clients' commands interleaving between them.

<img src="65dbd839-106f-4479-b351-b2767a52915b.png" width="400">



In [ ]:
# ------ CORE REDIS ------
import redis

# -- TTL & Expiration --
r = redis.Redis(
    host = '127.0.0.1',
    port = 6379,
    decode_responses=True
)
r.set("Name", "Pranit", ex=10) #-> Persist for 10 sec. This will expire in 10 sec
print(r.ttl("Name"))           #-> Check remaining persistance timing.
print(r.get("Name"))  



# -- Persistance --
#RDB -> Save snapshot based on interval and timing
r.set("Name", "Pranit")
r.set("City", "Mumbai")
# Trigger an RDB snapshot
print(r.bgsave())
print(r.config_get("dir"))
print(r.config_get("dbfilename"))
r.config_set("appendonly", "yes")
r.config_set("appendfsync", "everysec")
# Write some data
r.set("Name", "Pranit")
r.set("Age", 23)



# -- Pipeline -- 
with r.pipeline(transaction=False) as pipe:
    pipe.set("Name", "Pranit")
    pipe.set("City", "Mumbai")
    pipe.set("Age", 25)
    pipe.get("Name")
    result = pipe.execute()
print(result)



# -- Transaction --
r.set("account:A", 1000)
r.set("account:B", 500)
with r.pipeline() as pipe:
    pipe.multi()
    pipe.decrby("account:A", 200)
    pipe.incrby("account:B", 200)
    result = pipe.execute()
print(r.get("account:A"))  # 800
print(r.get("account:B"))  # 700

##### APPLICATION DEVELOPEMENT
- Connection Pooling : Using single connection for all processes to not connect again and again.
- Caching Strategies : 1.cache aside, 2.write thtough, 3.write behind
- Rate Limiting : Collect user requests and if requests are too many kill them by using r.expire(key, 60)
- Retry logic & backoff : So intead of retrying immediately, wait for some time and increase delays.